# Corporate Intelligence Analyst Agent
## Enterprise Agent Capstone Project

### The Pitch
**Problem:**  
Modern investors are drowning in data. Analyzing a single company requires synthesizing 10-K filings, earnings transcripts, real-time news, and market sentiment. For retail investors and busy professionals, doing this manually for every stock of interest is impossible.

**Solution:**  
The **Corporate Intelligence Analyst** is a multi-agent system that acts as an "Analyst Team in a Box." It autonomously gathers, analyzes, and synthesizes diverse data streams to produce a professional-grade, 3-paragraph investor summary in seconds.

**Value Proposition:**
- **Speed:** Reduces hours of research to seconds.
- **Objectivity:** Data-driven analysis without emotional bias.
- **Synthesis:** Connects the dots between hard numbers (Fundamentals) and soft signals (News/Sentiment).

### Architecture
The system follows a **"Newsroom"** architecture, orchestrated by a Manager Agent using the **Google Agent Development Kit (ADK)**.

```mermaid
graph TD
    User[User Input: Ticker Symbol] --> Manager[Manager Agent: Editor-in-Chief]
    Manager -->|Uses Tool| Quant[Quant Agent: Fundamentals]
    Manager -->|Uses Tool| Investigator[Investigator Agent: Risk & News]
    Manager -->|Uses Tool| Futurist[Futurist Agent: Outlook]
    
    Quant -->|Returns Financial Data| Manager
    Investigator -->|Returns Risk Report| Manager
    Futurist -->|Returns Experimental Forecast| Manager
    
    Manager -->|Synthesizes Final Report| Report[Final 3-Paragraph Summary]
```

### Key Concepts Demonstrated
1.  **Multi-Agent Orchestration:** Using `AgentTool` to enable agents to call other agents.
2.  **Tool Use:** Agents utilizing specific python functions and the built-in `google_search` tool.
3.  **Structured Output:** Enforcing a strict, professional reporting format.

## Setup & Configuration

In [ ]:
# Install necessary libraries
!pip install -q -U google-adk google-generativeai yfinance python-dotenv

In [33]:
import os
import logging
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, google_search
from google.genai import types

# Setup basic logging to see tool calls
logging.basicConfig(level=logging.INFO)

# Load API Key from .env file
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    print("⚠️ GOOGLE_API_KEY not found in environment. Please add it to your .env file.")
else:
    print("✅ API Key loaded successfully.")

✅ API Key loaded successfully.


## Core Tools with Logging
We've added print statements to these tools to help debug exactly what data is being passed around.

In [34]:
def get_fundamentals(ticker: str) -> str:
    """Fetches financial fundamentals for a given ticker."""
    print(f"\n[DEBUG] get_fundamentals called for: {ticker}")
    # Mock data
    data = "Revenue Growth: 15% YoY, Margins: 25% Operating Margin, Cash: High reserves, Debt: Low"
    print(f"[DEBUG] get_fundamentals returning: {data}")
    return data

def get_outlook(ticker: str) -> str:
    """Generates an experimental short-term outlook."""
    print(f"\n[DEBUG] get_outlook called for: {ticker}")
    # Mock data
    data = "Outlook: Bullish. Confidence: 63%. Reasoning: Strong earnings momentum outweighs regulatory concerns."
    print(f"[DEBUG] get_outlook returning: {data}")
    return data

# Wrapper for google_search to add logging
def search_tool(query: str) -> str:
    """Searches the web for information."""
    print(f"\n[DEBUG] search_tool called for: {query}")
    # Call the actual ADK tool
    # Note: google_search is a tool instance, we might need to invoke it or use it directly.
    # For simplicity in this debug phase, we'll just use it as is in the agent list,
    # but if we want to log it, we'd wrap it. 
    # Let's trust the agent to call 'google_search' directly for now, 
    # but we can inspect the Investigator's output in the test section.
    return "Search functionality active (logging wrapper not fully implemented for built-in tool)"

## Agents Setup (Text Output)
We have reverted to simple text output to ensure stability. We will test each agent individually.

In [35]:
MODEL_NAME = "gemini-2.5-flash"

# 1. Quant Agent
quant_agent = Agent(
    name="QuantAgent",
    model=MODEL_NAME,
    description="A financial analyst that provides fundamental data.",
    instruction="You are a financial analyst. Use the `get_fundamentals` tool to analyze the company's financial health. Return a concise text summary of the fundamentals.",
    tools=[get_fundamentals]
)

# 2. Investigator Agent
investigator_agent = Agent(
    name="InvestigatorAgent",
    model=MODEL_NAME,
    description="A risk investigator that finds news, regulatory issues, and sentiment.",
    instruction="""You are a risk investigator. Use the `google_search` tool to identify risks, regulatory issues, and market sentiment.
    Search for terms like 'lawsuits', 'regulatory investigation', 'scandal', and 'analyst sentiment'.
    Return a concise text summary of the risks and sentiment found.""",
    tools=[google_search]
)

# 3. Futurist Agent
futurist_agent = Agent(
    name="FuturistAgent",
    model=MODEL_NAME,
    description="A market futurist that provides an experimental short-term outlook.",
    instruction="You are a market futurist. Use the `get_outlook` tool to provide an experimental forecast. Return a concise text summary of the outlook.",
    tools=[get_outlook]
)

# 4. Manager Agent (The Orchestrator)
manager_agent = Agent(
    name="ManagerAgent",
    model=MODEL_NAME,
    instruction="""You are the Editor-in-Chief of an investment newsletter.
    Your goal is to produce a comprehensive 3-paragraph report on a company based on input from your team.
    
    Process:
    1. Call `QuantAgent` to get the financial fundamentals.
    2. Call `InvestigatorAgent` to get risks and news.
    3. Call `FuturistAgent` to get the outlook.
    4. Synthesize all information into a final report with exactly these three sections:
       - **Summary** (Fundamentals)
       - **Risks & Recent Developments** (News/Risks)
       - **Experimental Outlook** (Forecast)
    """,
    tools=[
        AgentTool(quant_agent),
        AgentTool(investigator_agent),
        AgentTool(futurist_agent)
    ]
)

## Orchestration Demo

In [39]:
# Run the full system
runner = InMemoryRunner(agent=manager_agent)

# Uncomment to run when API key is set in .env
await runner.run_debug("Analyze Apple Inc. (AAPL)")

INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False



 ### Created new session: debug_session_id

User > Analyze Apple Inc. (AAPL)


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False



[DEBUG] get_fundamentals called for: AAPL
[DEBUG] get_fundamentals returning: Revenue Growth: 15% YoY, Margins: 25% Operating Margin, Cash: High reserves, Debt: Low


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.


[DEBUG] get_outlook called for: AAPL
[DEBUG] get_outlook returning: Outlook: Bullish. Confidence: 63%. Reasoning: Strong earnings momentum outweighs regulatory concerns.


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.


ManagerAgent > **Summary**
Apple Inc. demonstrates strong financial health with a 15% year-over-year revenue growth and a solid 25% operating margin. The company maintains high cash reserves and low debt, indicating a robust financial position.

**Risks & Recent Developments**
Apple Inc. (AAPL) faces a range of significant risks, including numerous lawsuits, escalating regulatory scrutiny, and ongoing challenges related to its supply chain and product development. The company is currently battling antitrust lawsuits from the U.S. Department of Justice and concerns from the European Union regarding its App Store policies. Additionally, Apple is involved in intellectual property disputes, including a new ITC investigation into potential patent infringements by redesigned Apple Watches. Privacy and data collection concerns also persist, with a cybercrime probe by French prosecutors and a class-action lawsuit regarding employee device monitoring. Investor lawsuits allege misrepresentation 

[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Apple Inc. (AAPL) financial fundamentals'
         },
         id='adk-9f267e8f-dbec-4d89-a335-d502db0ef98c',
         name='QuantAgent'
       ),
       thought_signature=b'\n\x90\x03\x01\xd1\xed\x8ao\xf4.\xfer\xb0\xd7d\xb0X#\\\xa9\xd1q\xc1X[\x8c\xf8T\x0e"7/\x05\x0cs^F\x85\x07\xbd\x84\'\xc1\xa18\x1a$p\x0b\x0f\xde\x0f\x9e\x1a\xd4^OHfw\x98\xacR@\x83x\xa4\xfa\xbc\xc5\x8c\xfe\xd3?\xe9\xb1h\xe5\x7fK\xb4\xe6\x89\x9b\xdas\x00\xd8\xca\xf7I\xa7\xa3%\x94\x9f\xf3...'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=21,
   prompt_token_count=295,
   prompt_tokens_details=[
     ModalityTokenCount(
       mo